# RANZ VERSION 4 RDM Data Loader with Auto-Merge
## Enhanced with Table Validation

---

## What This Notebook Does

Loads RANZ V4 Reference Data Management (RDM) tables from Azure Data Lake Storage (ADLS) with:
* **Automatic LIVE + HISTORY merging** - Combines `_xref` (live) and `_hxrf` (history) tables
* **Smart deduplication** - Keeps latest records by primary key + timestamp
* **Data freshness validation** - Ensures LIVE data is not stale
* **Schema alignment** - Handles schema evolution automatically
* **NEW: Pre-validation** - Checks table existence before processing

---

## New Feature: Table Validation

### The Problem
Previously, if you passed an invalid table name like `'test'`, the function would only fail **during processing** when trying to access the table path.

### The Solution
**Pre-validation phase** that:
1. Checks if tables exist **before** starting any processing
2. Validates both `_xref` (LIVE) and `_hxrf` (HISTORY) variants for regular tables
3. Validates **static table name** (no suffix) for lookup tables (starting with `c_lkp_`)
4. Reports which tables are invalid with clear error messages
5. Offers two modes:
   * **Fail Fast** (default): Aborts if any invalid tables found
   * **Skip Invalid**: Skips invalid tables and continues with valid ones

---

## Usage

### Example 1: Fail Fast (Default)
```python
load_df = ['c_b_party', 'c_b_contract', 'c_lkp_naics', 'test']

# This will validate first, then ABORT if invalid tables found
results = load_ranz_v4_rdm_tables(load_df, '20260520')
```

**Output:**
```
[VALIDATION] Checking 4 table(s)...
  [OK] c_b_party: Valid (c_b_party_xref [OK], c_b_party_hxrf [OK])
  [OK] c_b_contract: Valid (c_b_contract_xref [OK], c_b_contract_hxrf [OK])
  [OK] c_lkp_naics: Valid (lookup table: c_lkp_naics [OK])
  [X] test: INVALID (neither test_xref nor test_hxrf exist)

[ERROR] Found 1 invalid table(s)
[ABORTED] Validation failed.
```

### Example 2: Skip Invalid Tables
```python
load_df = ['c_b_party', 'c_b_contract', 'c_lkp_naics', 'test']

# Skip invalid tables and continue
results = load_ranz_v4_rdm_tables(load_df, '20260520', skip_invalid=True)
```

**Output:**
```
[VALIDATION] Checking 4 table(s)...
  [OK] c_b_party: Valid (c_b_party_xref [OK], c_b_party_hxrf [OK])
  [OK] c_b_contract: Valid (c_b_contract_xref [OK])
  [OK] c_lkp_naics: Valid (lookup table: c_lkp_naics [OK])
  [X] test: INVALID (neither test_xref nor test_hxrf exist)

[WARNING] Skipping 1 invalid table(s) and continuing...

[PROCESSING] Starting data load...
  [OK] c_b_party: 125,430 rows
  [OK] c_b_contract: 89,234 rows
  [OK] c_lkp_naics: 3,456 rows
  [SKIP] test: SKIPPED - Invalid table
```

---

## Function Signature

```python
def load_ranz_v4_rdm_tables(
    load_df: list,           # List of table names
    load_date: str,          # Date in 'YYYYMMDD' format
    skip_invalid: bool = False  # NEW: Skip invalid tables?
) -> list:                   # Returns list of result dicts
```

**Parameters:**
* `load_df`: List of table names (can include `_xref` or `_hxrf` suffixes, or just base name)
* `load_date`: Date string in `YYYYMMDD` format (e.g., `'20260520'`)
* `skip_invalid`: *(NEW)*
  * `False` (default): Abort if any invalid tables found
  * `True`: Skip invalid tables and continue with valid ones

**Returns:**
* List of dictionaries with load results:
  ```python
  [
      {'table': 'c_b_party', 'status': 'SUCCESS', 'rows': 125430},
      {'table': 'c_b_contract', 'status': 'SUCCESS', 'rows': 89234},
      {'table': 'test', 'status': 'SKIPPED', 'reason': 'Invalid table'}
  ]
  ```

---

## Architecture

### Storage Structure
```
abfss://mdm-ranz@{storage_account}.dfs.core.windows.net/
├─ c_b_party_xref/4/data/
│  ├─ LOADED_DTS=20260518T120000Z/
│  ├─ LOADED_DTS=20260519T120000Z/
│  └─ LOADED_DTS=20260520T120000Z/
├─ c_b_party_hxrf/4/data/
│  └─ LOADED_DTS=20260520T120000Z/
└─ ...
```

### Load Modes

| Table Type | Suffix | Mode | Partition Strategy |
|------------|--------|------|--------------------|
| LIVE | `_xref` | Latest snapshot | Multi-partition (ONGOING_START to T-1) |
| HISTORY | `_hxrf` | Time-filtered | Single partition + HIST_CREATE_DATE filter |
| Lookup | `c_lkp_*` | Latest snapshot | No freshness check |

### Merge Logic
1. Read `{base_table}_xref` (LIVE)
2. Read `{base_table}_hxrf` (HISTORY)
3. Align schemas (add missing columns)
4. Union both DataFrames
5. Deduplicate by primary key + `EDL_ACT_DTS`
6. Create temp view with base table name

---

## Validation Details

### What Gets Validated
For each table in your list:
1. Extracts base table name (removes `_xref`/`_hxrf` suffix if present)
2. Checks table type:
   * **Regular tables** (`c_b_*`): At least one variant (`_xref` OR `_hxrf`) must exist
   * **Lookup tables** (`c_lkp_*`): Static table name (no suffix) must exist
3. Reports validation status for each table

### Table Naming Convention
* **Regular Tables:** `{base_name}_xref` and/or `{base_name}_hxrf`
  * Example: `c_b_party_xref`, `c_b_party_hxrf`
* **Lookup Tables:** `{base_name}` (no suffix)
  * Example: `c_lkp_naics`, `c_lkp_cdd_risk_rating`

### Validation Output
```
[VALIDATION] Checking 4 table(s)...
  [OK] c_b_party: Valid (c_b_party_xref [OK], c_b_party_hxrf [OK])
  [OK] c_b_contract: Valid (c_b_contract_xref [OK])
  [OK] c_lkp_naics: Valid (lookup table: c_lkp_naics [OK])
  [X] fake_table: INVALID (neither fake_table_xref nor fake_table_hxrf exist)
```

---

## Next Steps

See the following cells for:
1. **Cell 2**: Complete implementation with all helper functions
2. **Cell 3**: Usage examples with validation (fail fast vs skip invalid)
3. **Cell 4**: Validation logic explanation for different table types
4. **Cell 5**: Production example with valid tables only

---

## Requirements

* **Environment Variable**: `AU_GDP_Defined_Storage_Account` must be set
* **Storage Format**: Delta Lake
* **Partitioning**: By `LOADED_DTS={YYYYMMDD}T{HHmmss}Z`
* **Python Libraries**: `datetime`, `pyspark.sql`, `typing`, `re`, `os`

## Lookup Table Special Handling - Always Use Latest

### Problem Statement

Lookup tables (static reference data) are updated infrequently. You might have scenarios like:

* **Requested Date:** January 2022
* **Latest Lookup File:** December 2023
* **No files exist** between Jan 2022 and Dec 2023

With regular partition logic ("find closest partition on or before requested date"), this would **fail** because Dec 2023 is AFTER Jan 2022.

---

### Solution: Always Use Latest Available Partition

For lookup tables, the loader now:

1. **Ignores the requested date** for partition selection
2. **Always uses the LATEST available partition** regardless of how old or new it is
3. **No freshness validation** - lookup files can be months or years old

### Examples

#### Example 1: Lookup File is OLDER than Requested Date

```python
load_df = ['c_lkp_naics']
load_date = '20260520'  # Requesting May 2026 data

results = load_ranz_v4_rdm_tables(load_df, load_date)
```

**Available Partitions:**
* `LOADED_DTS=20231215T120000Z` (Dec 15, 2023)

**Output:**
```
[LOOKUP TABLE] Using latest available partition from 522 day(s) before requested date
  Requested: 20260520, Latest available: 20231215
```

**Result:** Uses Dec 2023 file even though it's 1.5 years old

---

#### Example 2: Lookup File is NEWER than Requested Date

```python
load_df = ['c_lkp_cdd_risk_rating']
load_date = '20220115'  # Requesting Jan 2022 data

results = load_ranz_v4_rdm_tables(load_df, load_date)
```

**Available Partitions:**
* `LOADED_DTS=20231220T120000Z` (Dec 20, 2023)

**Output:**
```
[LOOKUP TABLE] Using latest available partition from 705 day(s) AFTER requested date
  Requested: 20220115, Latest available: 20231220
```

**Result:** Uses Dec 2023 file even though it's almost 2 years NEWER

---

#### Example 3: Multiple Lookup Files Available

```python
load_df = ['c_lkp_legal_entity_type']
load_date = '20260520'

results = load_ranz_v4_rdm_tables(load_df, load_date)
```

**Available Partitions:**
* `LOADED_DTS=20231115T120000Z` (Nov 2023)
* `LOADED_DTS=20240308T120000Z` (Mar 2024)
* `LOADED_DTS=20250601T120000Z` (Jun 2025)

**Output:**
```
[LOOKUP TABLE] Using latest available partition from 353 day(s) before requested date
  Requested: 20260520, Latest available: 20250601
```

**Result:** Uses Jun 2025 file (most recent)

---

### Regular Tables (Non-Lookup) Behavior

Regular tables (business data) still use the **standard logic**:

* Finds partition **on or before** requested date
* Applies **freshness validation** (MAX_DAYS_OLD = 1 day for LIVE tables)
* **Fails if data is too stale**

```python
load_df = ['c_b_party']  # Regular table
load_date = '20260520'

# If latest partition is 20260518 (2 days old), it FAILS freshness check
# If latest partition is 20260519 (1 day old), it PASSES
```

---

### Summary

| Table Type | Partition Selection | Freshness Check | Example |
|------------|---------------------|-----------------|----------|
| **Lookup** (`c_lkp_*`) | Always LATEST available (ignores requested date) | None (can be years old) | Request 2022 data, use 2023 file |
| **Regular** (`c_b_*`) | Closest on or before requested date | MAX_DAYS_OLD = 1 day | Request 2026-05-20, use 2026-05-19 |

---

### Why This Makes Sense

Lookup tables contain:
* NAICS codes (industry classifications)
* Legal entity types
* Risk rating categories
* Country codes

These are **static reference data** that:
* Change rarely (months or years between updates)
* Don't have daily/weekly snapshots
* Should use the latest available definition regardless of historical analysis date

When analyzing 2022 transactions, you want the **current** NAICS code definitions (from 2023 file), not 2022 definitions (which might not even exist).

In [0]:
# ================================================================================
# RANZ VERSION 4 RDM Data Loader with Auto-Merge - OPTIMIZED
# ================================================================================
# ========== CONSTANTS ==========
CUTOFF_DATE = datetime.strptime('20260412', '%Y%m%d')
ONGOING_START = datetime.strptime('20260413', '%Y%m%d')
HISTORICAL_START = '2025-01-01'
 
# Data freshness validation (LIVE mode only)
MAX_DAYS_OLD = 1  # Maximum allowed staleness - job will FAIL if LIVE data is older
 
# Primary key mapping (using BASE table names without suffix)
PRIMARY_KEY_MAP = {
    "ROWID_OBJECT": [
        "c_b_party_rel_addr", "c_b_contract", "c_b_due_diligence",
        "c_b_party_dom_cntry", "c_b_party_naics", "c_b_party_pep_am",
        "c_b_party_rel_party", "c_lkp_cdd_risk_rating", "c_lkp_naics_sector",
        "c_lkp_naics", "c_b_contr_rol_party"
    ],
    "SRC_PARTY_ID": ["c_b_party"]
}
 
FORCE_LIVE_TABLES = ["c_b_business_line_xref", "c_rbo_rel_type_xref"]
 
# ========== HELPER FUNCTIONS ==========
def get_base_table_name(table_name: str) -> str:
    """Extract base table name by removing _xref or _hxrf suffix"""
    return table_name.lower().replace('_xref', '').replace('_hxrf', '')
 
def get_primary_key(base_table_name: str) -> Optional[str]:
    """Get primary key column for a table"""
    for pk_col, tables in PRIMARY_KEY_MAP.items():
        if base_table_name in tables:
            return pk_col
    return None
 
def deduplicate_dataframe(df: DataFrame, table_name: str) -> DataFrame:
    """Deduplicate DataFrame by primary key + EDL_ACT_DTS"""
    base_name = get_base_table_name(table_name)
    pk_column = get_primary_key(base_name)
   
    if not pk_column or pk_column not in df.columns or 'EDL_ACT_DTS' not in df.columns:
        return df
   
    before_count = df.count()
    window_spec = Window.partitionBy(pk_column).orderBy(col("EDL_ACT_DTS").desc())
    df_deduped = df.withColumn("rn", row_number().over(window_spec)).filter(col("rn") == 1).drop("rn")
    after_count = df_deduped.count()
   
    if before_count != after_count:
        print(f"Deduplication: {before_count:,} -> {after_count:,} rows (removed {before_count - after_count:,} duplicates)")
   
    return df_deduped
 
def align_schemas(df1: DataFrame, df2: DataFrame) -> tuple:
    """Align schemas of two DataFrames by adding missing columns"""
    cols1 = set(df1.columns)
    cols2 = set(df2.columns)
   
    if cols1 == cols2:
        return df1, df2
   
    all_cols = sorted(cols1.union(cols2))
   
    for col_name in all_cols:
        if col_name not in df1.columns:
            df1 = df1.withColumn(col_name, lit(None))
        if col_name not in df2.columns:
            df2 = df2.withColumn(col_name, lit(None))
   
    return df1.select(all_cols), df2.select(all_cols)
 
def build_base_path(table_name: str) -> str:
    """Build base path for table storage"""
    storage_account = os.environ.get('AU_GDP_Defined_Storage_Account')
    if not storage_account:
        raise ValueError("Environment variable 'AU_GDP_Defined_Storage_Account' not set")
    return f"abfss://mdm-ranz@{storage_account}.dfs.core.windows.net/{table_name}"

def validate_table_exists(table_name: str) -> tuple:
    """Validate if a table exists by checking its base path"""
    try:
        base_path = build_base_path(table_name)
        data_path = f"{base_path}/4/data/"
        
        # Try to list the path - if it exists, it will succeed
        dbutils.fs.ls(data_path)
        return (True, None)
    except Exception as e:
        error_msg = str(e)
        # Check if it's a "not found" error
        if "FileNotFoundException" in error_msg or "does not exist" in error_msg.lower():
            return (False, f"Table path does not exist: {table_name}")
        else:
            return (False, f"Error accessing table: {error_msg}")
 
def find_available_partitions(base_path: str) -> List[Dict]:
    """Find all available partitions for a table"""
    partitions = []
    data_path = f"{base_path}/4/data/"
   
    try:
        items = dbutils.fs.ls(data_path)
        for item in items:
            if item.isDir():
                partition_name = item.name.rstrip('/')
                match = re.search(r'LOADED_DTS=(\d{8})T\d{6}Z', partition_name)
                if match:
                    date_str = match.group(1)
                    date_obj = datetime.strptime(date_str, '%Y%m%d')
                    partitions.append({
                        'path': f"{data_path}{partition_name}/",
                        'date': date_obj,
                        'date_str': date_str
                    })
        partitions.sort(key=lambda x: x['date'])
    except Exception as e:
        print(f"Error finding partitions: {e}")
   
    return partitions

def is_lookup_table(table_name: str) -> bool:
    """Check if table is a lookup table by prefix (works with or without suffix)"""
    return table_name.lower().startswith("c_lkp_")

# ========== PARTITION SELECTION ==========
def select_live_partitions(partitions: List[Dict], requested_date: datetime, table_name: str) -> List[Dict]:
    """Select all partitions from ongoing_start to T-1 for LIVE mode"""
    t_minus_1 = requested_date - timedelta(days=1)
    selected = [p for p in partitions if ONGOING_START <= p['date'] <= t_minus_1]
   
    if not selected:
        raise ValueError(f"No partitions found between {ONGOING_START.strftime('%Y%m%d')} and {t_minus_1.strftime('%Y%m%d')}")
   
    # FRESHNESS CHECK (skip for lookup tables)
    if not is_lookup_table(table_name):
        latest_partition = selected[-1]
        days_diff = (t_minus_1 - latest_partition['date']).days

        if days_diff > MAX_DAYS_OLD:
            raise ValueError(
                f"Data freshness check FAILED: Latest partition is {days_diff} days old "
                f"(max allowed: {MAX_DAYS_OLD}). Expected partition up to "
                f"{t_minus_1.strftime('%Y%m%d')}, but latest is {latest_partition['date_str']}"
            )

    print(f"LIVE mode: Selected {len(selected)} partitions from {selected[0]['date_str']} to {selected[-1]['date_str']}")
    return selected

def find_closest_partition(partitions: List[Dict], requested_date: datetime, read_mode: str, table_name: str) -> Optional[Dict]:
    """Find closest partition on or before requested date (freshness check only for LIVE mode)
    
    For lookup tables: Always returns the LATEST available partition regardless of requested date
    For regular tables: Returns partition on or before requested date with freshness checks
    """
    # Special handling for lookup tables: always use latest partition
    if is_lookup_table(table_name):
        if not partitions:
            raise ValueError(f"No partitions found for lookup table {table_name}")
        
        latest_partition = sorted(partitions, key=lambda x: x['date'])[-1]
        days_diff = (requested_date - latest_partition['date']).days
        
        if days_diff < 0:
            print(f"[LOOKUP TABLE] Using latest available partition from {abs(days_diff)} day(s) AFTER requested date")
            print(f"  Requested: {requested_date.strftime('%Y%m%d')}, Latest available: {latest_partition['date_str']}")
        elif days_diff == 0:
            print(f"[LOOKUP TABLE] Using partition from requested date")
        else:
            print(f"[LOOKUP TABLE] Using latest available partition from {days_diff} day(s) before requested date")
            print(f"  Requested: {requested_date.strftime('%Y%m%d')}, Latest available: {latest_partition['date_str']}")
        
        return latest_partition
    
    # Regular tables: find partition on or before requested date
    valid_partitions = [p for p in partitions if p['date'] <= requested_date]
    if not valid_partitions:
        raise ValueError(f"No partition found on or before {requested_date.strftime('%Y%m%d')}")
   
    selected_partition = sorted(valid_partitions, key=lambda x: x['date'])[-1]
    days_diff = (requested_date - selected_partition['date']).days
   
    # FRESHNESS CHECK - Only for LIVE mode and non-lookup tables
    if read_mode == 'LIVE' and days_diff > MAX_DAYS_OLD:
        raise ValueError(
            f"Data freshness check FAILED: Latest partition is {days_diff} days old (max allowed: {MAX_DAYS_OLD}). "
            f"Requested date: {requested_date.strftime('%Y%m%d')}, Latest partition: {selected_partition['date_str']}"
        )
   
    if days_diff > 0:
        if read_mode == 'LIVE':
            print(f"Using partition from {days_diff} day(s) before requested date (within tolerance)")
        else:
            print(f"Using partition from {days_diff} day(s) before requested date")
   
    return selected_partition
 
# ========== DATA READING ==========
def read_multiple_partitions(partitions: List[Dict], table_name: str) -> DataFrame:
    """Read and combine multiple partitions for LIVE mode"""
    # Read all partitions and union
    dfs = [spark.read.format('delta').load(p['path']) for p in partitions]
    df_combined = dfs[0]
    for df in dfs[1:]:
        df_combined = df_combined.union(df)
   
    total_rows = df_combined.count()
    print(f"\nCombined total: {total_rows:,} rows from {len(partitions)} partitions")
   
    # Deduplicate
    df_deduped = deduplicate_dataframe(df_combined, table_name)
    final_count = df_deduped.count()
    print(f"Final row count: {final_count:,}")
   
    return df_deduped
 
def apply_history_date_filter(df: DataFrame, requested_datetime: datetime) -> DataFrame:
    """Apply HIST_CREATE_DATE filtering for HISTORY tables"""
    if 'HIST_CREATE_DATE' not in df.columns:
        return df
   
    if requested_datetime <= CUTOFF_DATE:
        # One-time load
        end_date = requested_datetime.strftime('%Y-%m-%d')
        print(f"HISTORY mode (one-time): Filtering HIST_CREATE_DATE from {HISTORICAL_START} to {end_date}")
        return df.filter((col('HIST_CREATE_DATE') >= HISTORICAL_START) & (col('HIST_CREATE_DATE') <= end_date))
    else:
        # Ongoing load: Historical + Live segments
        hist_end = CUTOFF_DATE.strftime('%Y-%m-%d')
        df_historical = df.filter((col('HIST_CREATE_DATE') >= HISTORICAL_START) & (col('HIST_CREATE_DATE') <= hist_end))
       
        t_minus_1 = requested_datetime - timedelta(days=1)
        live_start = ONGOING_START.strftime('%Y-%m-%d')
        live_end = t_minus_1.strftime('%Y-%m-%d') if t_minus_1 >= ONGOING_START else live_start
       
        df_live = df.filter((col('HIST_CREATE_DATE') >= live_start) & (col('HIST_CREATE_DATE') <= live_end))
        print(f"HISTORY mode (ongoing): Historical + Live segments merged")
        return df_historical.union(df_live)
 
def read_single_partition(partition: Dict, table_name: str, read_mode: str, requested_date: str) -> DataFrame:
    """Read single partition with optional filtering"""
    df = spark.read.format('delta').load(partition['path'])
    initial_count = df.count()
    print(f"Initial row count: {initial_count:,}")
   
    # Apply filtering based on read mode
    if read_mode == 'LOOKUP':
        print(f"LOOKUP mode: Reading complete snapshot from partition (no filtering)")
    elif read_mode == 'HISTORY':
        requested_datetime = datetime.strptime(requested_date, '%Y%m%d')
        df = apply_history_date_filter(df, requested_datetime)
    else:  # LIVE mode
        print(f"LIVE mode: Reading complete snapshot from partition")
   
    # Deduplicate
    df_deduped = deduplicate_dataframe(df, table_name)
    final_count = df_deduped.count()
    print(f"Final row count: {final_count:,}")
   
    return df_deduped
 
# ========== MAIN READ FUNCTION ==========
def read_ranz_v4_data(table_name: str, requested_date: str) -> DataFrame:
    """Main entry point to read RANZ V4 data"""
    print(f"\n{'='*80}")
    print(f"RANZ-MDM: Processing {table_name} for date {requested_date}")
    print(f"{'='*80}")
   
    # Validate inputs
    if not table_name or not requested_date:
        raise ValueError("Table name and date are mandatory")
    if not re.match(r'^\d{8}$', requested_date):
        raise ValueError(f"Invalid date format: {requested_date}. Expected YYYYMMDD")
   
    requested_datetime = datetime.strptime(requested_date, '%Y%m%d')
   
    # Determine read mode (check for lookup tables first)
    if is_lookup_table(table_name):
        read_mode = 'LOOKUP'
    else:
        table_lower = table_name.lower()
        is_live = table_lower.endswith('_xref') or table_lower in FORCE_LIVE_TABLES
        read_mode = 'LIVE' if is_live else 'HISTORY'
    
    print(f"Read mode: {read_mode}")
   
    # Find partitions
    base_path = build_base_path(table_name)
    available_partitions = find_available_partitions(base_path)
   
    if not available_partitions:
        raise ValueError(f"No data found for table {table_name}")
    print(f"Available partitions: {len(available_partitions)} found")
   
    # Read data based on mode and date
    if read_mode == 'LIVE' and requested_datetime >= ONGOING_START:
        # Multi-partition read for LIVE tables from cutoff onwards
        selected_partitions = select_live_partitions(available_partitions, requested_datetime,table_name)
        return read_multiple_partitions(selected_partitions, table_name)
    else:
        # Single partition read for HISTORY or LIVE before cutoff
        selected_partition = find_closest_partition(available_partitions, requested_datetime, read_mode, table_name)
        print(f"Selected partition date: {selected_partition['date_str']}")
        return read_single_partition(selected_partition, table_name, read_mode, requested_date)
 
# ========== MERGE FUNCTION ==========
def merge_live_and_history(base_table_name: str, requested_date: str) -> DataFrame:
    """Merge LIVE and HISTORY tables (or read static lookup table)"""
    print(f"\n{'='*80}")
    print(f"PROCESSING: {base_table_name}")
    print(f"{'='*80}")
   
    # Check if this is a lookup table (only _xref, no _hxrf)
    if is_lookup_table(base_table_name):
        live_table = f"{base_table_name}_xref"
        print(f"Lookup table detected - reading {live_table}")
        df = read_ranz_v4_data(live_table, requested_date)
        row_count = df.count()
        print(f"Lookup table row count: {row_count:,}")
        return df
   
    # Regular tables: merge _xref and _hxrf
    print(f"Regular table detected - merging LIVE and HISTORY variants")
    live_table = f"{base_table_name}_xref"
    history_table = f"{base_table_name}_hxrf"
   
    # Read both tables
    df_live, live_count = None, 0
    df_history, history_count = None, 0
   
    try:
        df_live = read_ranz_v4_data(live_table, requested_date)
        live_count = df_live.count()
    except Exception as e:
        print(f"LIVE table failed: {e}")
   
    try:
        df_history = read_ranz_v4_data(history_table, requested_date)
        history_count = df_history.count()
    except Exception as e:
        print(f"HISTORY table failed: {e}")
   
    # Handle failures
    if df_live is None and df_history is None:
        raise ValueError("Both tables failed")
    if df_live is None:
        return df_history
    if df_history is None:
        return df_live
   
    # Align schemas and union
    df_live, df_history = align_schemas(df_live, df_history)
    df_merged = df_live.union(df_history)
    merged_count = df_merged.count()
    print(f"Merged: {live_count:,} + {history_count:,} = {merged_count:,}")
   
    # Deduplicate across both tables
    pk_column = get_primary_key(base_table_name)
    if pk_column:
        print(f"Deduplicating merged data using primary key: {pk_column}")
        df_merged = deduplicate_dataframe(df_merged, base_table_name)
    else:
        print(f"No deduplication applied - primary key not found")
   
    return df_merged
 
# ========== MAIN LOAD FUNCTION ==========
def load_ranz_v4_rdm_tables(load_df: list, load_date: str, skip_invalid: bool = False):
    """
    Load RANZ V4 RDM tables with automatic LIVE and HISTORY merging.
   
    Args:
        load_df: List of table names (can include _xref or _hxrf suffixes)
        load_date: Date string in 'YYYYMMDD' format (e.g., '20260520')
        skip_invalid: If True, skip invalid tables and continue; if False, fail fast (default: False)
   
    Returns:
        List of result dictionaries with status and row counts
    """
    print(f"\n{'='*80}")
    print(f"LOADING RDM DATA OBJECTS - Load Date: {load_date}")
    print(f"{'='*80}")
    
    # ========== VALIDATION PHASE ==========
    print(f"\n[VALIDATION] Checking {len(load_df)} table(s)...")
    
    validation_results = {}
    invalid_tables = []
    
    for dataobject in load_df:
        base_name = get_base_table_name(dataobject)
        is_lookup = is_lookup_table(base_name)
        
        # Lookup tables have _xref only (no _hxrf), regular tables have _xref and/or _hxrf
        if is_lookup:
            # Lookup tables: check only _xref variant (no _hxrf exists for lookup tables)
            live_table = f"{base_name}_xref"
            live_exists, live_error = validate_table_exists(live_table)
            
            validation_results[base_name] = {
                'live_exists': live_exists,
                'live_error': live_error,
                'is_lookup': True
            }
            
            if not live_exists:
                invalid_tables.append({
                    'table': base_name,
                    'live_table': live_table,
                    'live_error': live_error,
                    'is_lookup': True
                })
                print(f"  [X] {base_name}: INVALID ({live_table} does not exist)")
            else:
                print(f"  [OK] {base_name}: Valid (lookup table: {live_table} [OK])")
        else:
            # Regular tables: check both _xref and _hxrf variants
            live_table = f"{base_name}_xref"
            history_table = f"{base_name}_hxrf"
            
            live_exists, live_error = validate_table_exists(live_table)
            history_exists, history_error = validate_table_exists(history_table)
            
            validation_results[base_name] = {
                'live_exists': live_exists,
                'history_exists': history_exists,
                'live_error': live_error,
                'history_error': history_error,
                'is_lookup': False
            }
            
            # At least one variant (_xref or _hxrf) must exist
            if not live_exists and not history_exists:
                invalid_tables.append({
                    'table': base_name,
                    'live_table': live_table,
                    'history_table': history_table,
                    'live_error': live_error,
                    'history_error': history_error,
                    'is_lookup': False
                })
                print(f"  [X] {base_name}: INVALID (neither {live_table} nor {history_table} exist)")
            else:
                status_parts = []
                if live_exists:
                    status_parts.append(f"{live_table} [OK]")
                if history_exists:
                    status_parts.append(f"{history_table} [OK]")
                print(f"  [OK] {base_name}: Valid ({', '.join(status_parts)})")
    
    # Handle invalid tables
    if invalid_tables:
        print(f"\n{'='*80}")
        print(f"[ERROR] Found {len(invalid_tables)} invalid table(s):")
        print(f"{'='*80}")
        
        for inv in invalid_tables:
            print(f"\n  Table: {inv['table']}")
            if inv.get('is_lookup', False):
                print(f"{inv['live_table']}: {inv['live_error']}")
                print(f"    Note: Lookup tables have only _xref variant (no _hxrf)")
            else:
                print(f"{inv['live_table']}: {inv['live_error']}")
                print(f"{inv['history_table']}: {inv['history_error']}")
        
        if not skip_invalid:
            print(f"\n{'='*80}")
            print(f"[ABORTED] Validation failed. Set skip_invalid=True to skip invalid tables.")
            print(f"{'='*80}")
            raise ValueError(f"Invalid tables found: {[t['table'] for t in invalid_tables]}. Validation failed.")
        else:
            print(f"\n[WARNING] Skipping {len(invalid_tables)} invalid table(s) and continuing...")
    else:
        print(f"\n[VALIDATION] All {len(load_df)} table(s) are valid!")
    
    print(f"\n{'='*80}")
    print(f"[PROCESSING] Starting data load...")
    print(f"{'='*80}")
   
    processed = set()
    results = []
   
    for dataobject in load_df:
        base_name = get_base_table_name(dataobject)
       
        # Skip if already processed
        if base_name in processed:
            continue
        
        # Skip if invalid (when skip_invalid=True)
        if skip_invalid and base_name in [t['table'] for t in invalid_tables]:
            results.append({'table': base_name, 'status': 'SKIPPED', 'reason': 'Invalid table'})
            print(f"\n[SKIPPED] {base_name} - Invalid table")
            continue
       
        try:
            df_merged = merge_live_and_history(base_name, load_date)
            df_merged.createOrReplaceTempView(base_name)
            row_count = df_merged.count()
            processed.add(base_name)
            results.append({'table': base_name, 'status': 'SUCCESS', 'rows': row_count})
            print(f"\n[SUCCESS] Created view: {base_name} with {row_count:,} rows")
        except Exception as e:
            results.append({'table': base_name, 'status': 'FAILED', 'error': str(e)})
            print(f"\n[FAILED] {base_name} - {e}")    
 
   
    return results

## Validation Logic for Different Table Types

### Regular Tables (Business Data)
**Examples:** `c_b_party`, `c_b_contract`, `c_b_due_diligence`

* **Validation Rule:** At least ONE variant must exist (`_xref` OR `_hxrf`)
* **Why:** Business tables can have LIVE data only, HISTORY data only, or both
* **Checked Variants:**
  * `{table_name}_xref` (LIVE snapshot)
  * `{table_name}_hxrf` (HISTORY time-series)

**Validation Output:**
```
[OK] c_b_party: Valid (c_b_party_xref [OK], c_b_party_hxrf [OK])
[OK] c_b_contract: Valid (c_b_contract_xref [OK])
```

---

### Lookup Tables (Static Reference Data)
**Examples:** `c_lkp_cdd_risk_rating`, `c_lkp_naics`, `c_lkp_legal_entity_type`

* **Validation Rule:** Only `_xref` variant must exist (no `_hxrf` for lookup tables)
* **Why:** Lookup tables are static reference data that only have LIVE snapshots
* **Checked Variants:**
  * `{table_name}_xref` - **REQUIRED** (LIVE snapshot only)
  * `{table_name}_hxrf` - Does NOT exist for lookup tables

**Validation Output:**
```
[OK] c_lkp_cdd_risk_rating: Valid (lookup table: c_lkp_cdd_risk_rating_xref [OK])
[OK] c_lkp_naics: Valid (lookup table: c_lkp_naics_xref [OK])
```

---

### How It Detects Lookup Tables
Any table whose **base name** starts with `c_lkp_` is treated as a lookup table:
```python
def is_lookup_table(table_name: str) -> bool:
    return get_base_table_name(table_name).startswith("c_lkp_")
```

---

### Summary Table

| Table Type | Prefix | Table Name Format | Required Variants | Example |
|------------|--------|-------------------|-------------------|----------|
| **Regular** | `c_b_*` | Base name + suffix | At least one: `_xref` OR `_hxrf` | `c_b_party_xref`, `c_b_party_hxrf` |
| **Lookup** | `c_lkp_*` | Base name + suffix | Only `_xref` (no `_hxrf`) | `c_lkp_naics_xref` |
| **Force LIVE** | (in list) | Base name + suffix | At least one: `_xref` OR `_hxrf` | `c_b_business_line_xref` |

---

### Example: Mixed Table List
```python
load_df = [
    'c_b_party',              # Regular: checks c_b_party_xref and c_b_party_hxrf
    'c_b_contract_xref',      # Regular: checks c_b_contract_xref and c_b_contract_hxrf
    'c_lkp_naics',            # Lookup: checks c_lkp_naics_xref only
    'c_lkp_cdd_risk_rating',  # Lookup: checks c_lkp_cdd_risk_rating_xref only
    'test'                    # Invalid: neither variant exists
]

results = load_ranz_v4_rdm_tables(load_df, '20260520')
```

**Output:**
```
[VALIDATION] Checking 5 table(s)...
  [OK] c_b_party: Valid (c_b_party_xref [OK], c_b_party_hxrf [OK])
  [OK] c_b_contract: Valid (c_b_contract_xref [OK])
  [OK] c_lkp_naics: Valid (lookup table: c_lkp_naics_xref [OK])
  [OK] c_lkp_cdd_risk_rating: Valid (lookup table: c_lkp_cdd_risk_rating_xref [OK])
  [X] test: INVALID (neither test_xref nor test_hxrf exist)

[ERROR] Found 1 invalid table(s)
```

## Table Naming Convention Examples

### Storage Structure in ADLS

```
abfss://mdm-ranz@{storage_account}.dfs.core.windows.net/

REGULAR TABLES (with _xref and _hxrf suffixes):
├─ c_b_party_xref/4/data/
│  ├─ LOADED_DTS=20260518T120000Z/
│  ├─ LOADED_DTS=20260519T120000Z/
│  └─ LOADED_DTS=20260520T120000Z/
├─ c_b_party_hxrf/4/data/
│  └─ LOADED_DTS=20260520T120000Z/
├─ c_b_contract_xref/4/data/
│  └─ LOADED_DTS=20260520T120000Z/
├─ c_b_contract_hxrf/4/data/
│  └─ LOADED_DTS=20260520T120000Z/

LOOKUP TABLES (only _xref variant, no _hxrf):
├─ c_lkp_naics_xref/4/data/
│  └─ LOADED_DTS=20260520T120000Z/
├─ c_lkp_cdd_risk_rating_xref/4/data/
│  └─ LOADED_DTS=20260520T120000Z/
└─ c_lkp_legal_entity_type_xref/4/data/
   └─ LOADED_DTS=20260520T120000Z/
```

---

### Example: Loading Mixed Tables

```python
load_df = [
    # Regular tables (function checks both _xref and _hxrf)
    'c_b_party',              # Looks for: c_b_party_xref, c_b_party_hxrf
    'c_b_contract_xref',      # Looks for: c_b_contract_xref, c_b_contract_hxrf
    'c_b_contr_rol_party',    # Looks for: c_b_contr_rol_party_xref, c_b_contr_rol_party_hxrf
    
    # Lookup tables (function checks only _xref variant)
    'c_lkp_naics',            # Looks for: c_lkp_naics_xref only (no _hxrf)
    'c_lkp_cdd_risk_rating',  # Looks for: c_lkp_cdd_risk_rating_xref only
    'c_lkp_legal_entity_type' # Looks for: c_lkp_legal_entity_type_xref only
]

results = load_ranz_v4_rdm_tables(load_df, '20260520')
```

---

### What Happens During Load

#### Regular Table: `c_b_party`
1. **Validation:** Checks if `c_b_party_xref` OR `c_b_party_hxrf` exist
2. **Loading:** Reads both variants (if they exist)
3. **Merging:** Unions the two DataFrames
4. **Deduplication:** Keeps latest record by primary key + EDL_ACT_DTS
5. **Result:** Creates temp view `c_b_party`

#### Lookup Table: `c_lkp_naics`
1. **Validation:** Checks if `c_lkp_naics_xref` exists (only _xref, no _hxrf)
2. **Loading:** Reads `c_lkp_naics_xref` directly
3. **No Merging:** Single table (no _hxrf variant to merge)
4. **Deduplication:** Applied if primary key exists
5. **Result:** Creates temp view `c_lkp_naics`

---

### Common Mistakes to Avoid

| Mistake | Incorrect | Correct |
|---------|-----------|----------|
| Missing prefix detection | `'lkp_naics'` | `'c_lkp_naics'` |
| Wrong assumption about storage | Assuming `c_lkp_naics` file exists | It's actually `c_lkp_naics_xref` in storage |

**Note:** You can pass table names with or without suffix:
* `'c_b_party'` → checks `c_b_party_xref` and `c_b_party_hxrf`
* `'c_b_party_xref'` → checks `c_b_party_xref` and `c_b_party_hxrf`
* `'c_lkp_naics'` → checks `c_lkp_naics_xref` only (no `_hxrf` for lookups)
* `'c_lkp_naics_xref'` → checks `c_lkp_naics_xref` only

The function automatically strips suffixes and determines the correct variants to check.

In [0]:
# ================================================================================
# PRODUCTION EXAMPLE - Clean Run with Valid Tables
# ================================================================================

# Import required libraries (if not already imported)
from datetime import datetime, timedelta
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, lit, row_number
from pyspark.sql.window import Window
from typing import List, Dict, Optional
import re
import os

# Set load date
Load_Date = '20260520'

# Define VALID tables to load
load_df_valid = [
    'c_b_party',                    # Party master
    'c_b_contract',                 # Contract data
    'c_b_party_rel_party',          # Party relationships
    'c_b_due_diligence',            # Due diligence records
    'c_lkp_cdd_risk_rating',        # Lookup: CDD risk ratings
    'c_lkp_naics'                   # Lookup: NAICS codes
]

print("="*80)
print("PRODUCTION LOAD - Valid Tables Only")
print("="*80)
print(f"Tables to load: {len(load_df_valid)}")
print(f"Load date: {Load_Date}")
print("="*80)

# Run the load (will validate first, then process)
results = load_ranz_v4_rdm_tables(load_df_valid, Load_Date)

# Display summary
print("\n" + "="*80)
print("LOAD SUMMARY")
print("="*80)

success_count = sum(1 for r in results if r['status'] == 'SUCCESS')
failed_count = sum(1 for r in results if r['status'] == 'FAILED')
total_rows = sum(r.get('rows', 0) for r in results if r['status'] == 'SUCCESS')

print(f"\nTotal tables: {len(results)}")
print(f"  [OK] Successful: {success_count}")
print(f"  [X] Failed: {failed_count}")
print(f"\nTotal rows loaded: {total_rows:,}")

print("\nDetailed Results:")
print("-" * 80)
for r in results:
    status = r['status']
    table = r['table']
    if status == 'SUCCESS':
        print(f"  [OK] {table:40} {r['rows']:>15,} rows")
    else:
        print(f"  [X] {table:40} FAILED: {r.get('error', 'Unknown')}")

print("\n" + "="*80)

# Show available temp views
print("\nAvailable temp views (can be queried with spark.sql):")
for r in results:
    if r['status'] == 'SUCCESS':
        print(f"  - {r['table']}")

In [0]:
# ================================================================================
# QUICK TEST: Verify Lookup Table Validation Fix
# ================================================================================

from datetime import datetime, timedelta
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, lit, row_number
from pyspark.sql.window import Window
from typing import List, Dict, Optional
import re
import os

Load_Date = '20260609'

print("="*80)
print("TESTING: Lookup Table Validation - c_lkp_naics")
print("="*80)
print()
print("Before Fix: Validation checked for 'c_lkp_naics' (WRONG)")
print("After Fix:  Validation checks for 'c_lkp_naics_xref' (CORRECT)")
print()
print("="*80)

# Test with lookup table only
load_df = ['c_lkp_naics']

try:
    results = load_ranz_v4_rdm_tables(load_df, Load_Date)
    
    print("\n" + "="*80)
    print("✓ SUCCESS - Lookup table validated and loaded!")
    print("="*80)
    
    for r in results:
        if r['status'] == 'SUCCESS':
            print(f"\nTable: {r['table']}")
            print(f"Rows: {r['rows']:,}")
            print(f"Status: {r['status']}")
            
            # Query sample data
            print("\nSample data:")
            df = spark.sql(f"SELECT * FROM {r['table']} LIMIT 5")
            display(df)
        else:
            print(f"\n✗ {r['table']}: {r['status']} - {r.get('error', 'Unknown')}")
            
except ValueError as e:
    print("\n" + "="*80)
    print("✗ VALIDATION FAILED")
    print("="*80)
    print(f"Error: {e}")
    print("\nPossible causes:")
    print("  1. c_lkp_naics_xref does not exist in storage")
    print("  2. Azure storage credentials not configured")
    print("  3. Environment variable 'AU_GDP_Defined_Storage_Account' not set")
except Exception as e:
    print("\n" + "="*80)
    print("✗ UNEXPECTED ERROR")
    print("="*80)
    print(f"Error: {e}")